# Getting started with TDatum

Run this notebook with a **SageMath kernel**, after installing the package
with `sage -pip install --no-deps .` from the source checkout.

A T-datum consists of polynomial matrices $A_+,A_-$ and a positive integer
diagonal matrix $D$ satisfying the conditions in
[Mizuno, *Difference equations arising from cluster algebras*](https://arxiv.org/abs/1912.05710).
The constructor checks these conditions for the supplied matrices.


In [1]:
from sage.all import QQ, LaurentPolynomialRing, matrix, diagonal_matrix
from tdatum import TDatum

R = LaurentPolynomialRing(QQ, "z")
z = R.gen()
A_plus = matrix(R, [[1 + z**2, 0], [0, 1 + z**2]])
A_minus = matrix(R, [[1 + z**2, -z], [-z, 1 + z**2]])
td = TDatum(A_plus, A_minus)
td.triple()

(
[1 + z^2       0]  [0 0]  [0 z]
[      0 1 + z^2], [0 0], [z 0]
)

The triple is $(N_0,N_+,N_-)$, where $A_+=N_0-N_+$ and $A_-=N_0-N_-$.

In [2]:
N0, Np, Nm = td.triple()
assert A_plus == N0 - Np
assert A_minus == N0 - Nm
assert td.degrees() == (2, 2)
assert td.symmetrizer() == (1, 1)
{"degrees": td.degrees(), "symmetrizer": td.symmetrizer(), "permutation": td.permutation()}

{'degrees': (2, 2), 'symmetrizer': (1, 1), 'permutation': [1, 2]}

## A nonidentity symmetrizer

For the following nonsymmetric matrix $A_-$, specify $D=\operatorname{diag}(1,2)$.
Writing $A^\dagger=(A(z^{-1}))^{\mathsf T}$, the symplectic relation is
$A_+DA_-^\dagger=A_-DA_+^\dagger$.


In [3]:
A_minus = matrix(R, [[1 + z**2, -z], [-2*z, 1 + z**2]])
D = diagonal_matrix([1, 2])
td = TDatum(A_plus, A_minus, D)
assert A_plus * D * A_minus.transpose().subs({z: 1/z}) == A_minus * D * A_plus.transpose().subs({z: 1/z})
td.symmetrizer()

(1, 2)

The sign dual exchanges $A_+$ and $A_-$ and keeps $D$. The Langlands dual conjugates the matrices by $D$ and changes the symmetrizer as described in the reference.

In [4]:
dual = td.sign_dual()
assert dual.symmetrizer() == (1, 2)
assert dual.sign_dual().pair() == td.pair()
assert td.langlands_dual().langlands_dual().pair() == td.pair()
dual.pair()

(
[1 + z^2      -z]  [1 + z^2       0]
[   -2*z 1 + z^2], [      0 1 + z^2]
)

## Validation failure

The constructor rejects a noninteger polynomial coefficient. Exact validation of a T-datum does not establish periodicity of its T/Y-system.

In [5]:
try:
    TDatum(matrix(R, [[1 + z**2]]), matrix(R, [[1 + z**2 - z/2]]))
except ValueError as error:
    print(error)
else:
    raise AssertionError("A noninteger coefficient was accepted")

A_plus and A_minus must have integer coefficients.
